# Model Evaluation: `mlflow.evaluate()`, custom metrics, and validation gates

This is the second **advanced** tutorial in the repo. It assumes you've worked through:

- `b_tracking_quickstart.ipynb` — one model, one run, one logged artifact.
- `a_hyperparameter_tuning.ipynb` — parent + child runs, an Optuna sweep, the registry.
- `b_logging_plots.ipynb` — hand-rolled diagnostic plots (residuals, Q–Q, predicted-vs-actual) logged with `mlflow.log_figure(...)`.
- `c_logging_callbacks.ipynb` — Optuna callbacks, parent metric history, per-trial model logging.

`d_` already showed you what a careful model evaluation looks like when you write everything yourself: a `compute_metrics(...)` helper, a small library of `plot_*` functions, then a per-trial pass that logs all of it. That code is fine, but it is also *yours to maintain forever* — every project rewrites slight variations of the same residual plot and the same `r2_score` call.

`mlflow.evaluate()` is MLflow's answer: pass a logged model and a held-out DataFrame, get a consistent set of metrics + diagnostic artifacts attached to the run automatically. This notebook walks through:

1. The minimal one-line call (the **beginner** story).
2. Adding a **custom metric** with `make_metric` for KPIs that matter more to your stakeholders than RMSE.
3. Adding a **custom artifact** for plots the default evaluator doesn't produce.
4. **Validation thresholds** — turning evaluation into a *promotion gate* that fails the run if the model regresses (CI for ML models).


## Source material

This notebook is a reorganized, expanded version of MLflow's official model-evaluation material. The primary sources used while drafting:

- **[ML Model Evaluation overview](https://mlflow.org/docs/latest/ml/evaluation/)** — the top-level page describing what the evaluation framework covers and how traditional-ML evaluation differs from `mlflow.genai.evaluate`.
- **[Model Evaluation (3.1.3)](https://mlflow.org/docs/3.1.3/ml/evaluation/model-eval/)** — the canonical regressor/classifier walkthroughs the *minimal call* in Step 3 is adapted from.
- **[Custom Metrics & Visualizations (3.1.3)](https://mlflow.org/docs/3.1.3/ml/evaluation/metrics-visualizations/)** — `make_metric`, `MetricValue`, and the `custom_artifacts=[...]` signature used in Steps 4–6.
- **[`evaluate_with_custom_metrics.py`](https://github.com/mlflow/mlflow/blob/master/examples/evaluation/evaluate_with_custom_metrics.py)** — the upstream regression example for an `extra_metrics` + `custom_artifacts` workflow.
- **[`evaluate_with_model_validation.py`](https://github.com/mlflow/mlflow/blob/master/examples/evaluation/evaluate_with_model_validation.py)** — the upstream end-to-end example for `MetricThreshold` and `validate_evaluation_results`; the candidate-vs-baseline pattern mentioned in Step 7 comes from here.
- **[`mlflow.models.evaluate` API reference](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.evaluate)** — full parameter list, including the deprecation note for the top-level `mlflow.evaluate` alias.
- **[`evaluate_with_custom_metrics_comprehensive.py`](https://github.com/mlflow/mlflow/blob/master/examples/evaluation/evaluate_with_custom_metrics_comprehensive.py)** — a longer upstream example with multiple custom metrics and a more elaborate custom-artifacts function, worth reading if you want patterns beyond what Steps 4–6 cover.

Where this notebook **diverges from upstream** (different dataset, sklearn instead of XGBoost in Step 2, the `within_50k` business metric, the residuals-by-income-decile artifact, the "use `mlflow.models.evaluate` not `mlflow.evaluate`" callout), the divergence is named in the surrounding prose or in a `NOTE (differs from upstream)` comment, following the convention used in `b_tracking_quickstart.ipynb`.


## What `mlflow.models.evaluate()` is and what it replaces

`mlflow.models.evaluate(model_uri, eval_data, targets=..., model_type="regressor")` is a single call that:

- loads the **already-logged** model back from MLflow (so evaluation is tied to the artifact, not to whatever lives in memory),
- runs it against the held-out frame,
- logs a standard set of metrics and diagnostic artifacts to the **active run**.

Contrast with what `d_` did by hand:

| Concern | Hand-rolled (`d_`) | `mlflow.models.evaluate()` |
|---|---|---|
| Standard metrics (MAE, MSE, RMSE, R², …) | `compute_metrics(...)` helper you maintain | `evaluators=["default"]` |
| Residual / predicted-vs-actual plots | Five `plot_*` functions + `log_figure` calls | Built-in artifacts (`shap_*`, `eval_results_table`, etc.) |
| Re-evaluating last quarter's model on new data | Re-instantiate, refit pipeline, re-import metrics | Pass the same `model_uri`, swap `eval_data` |
| Promotion gate ("only register if RMSE < X") | An `if` in your notebook nobody else sees | `MetricThreshold` + `validate_evaluation_results` |
| Custom business metric (e.g. *fraction of predictions within ±$50K*) | Helper function you call manually | `make_metric(...)` + `extra_metrics=[...]` |

The point isn't that the hand-rolled approach is wrong — it's that **`mlflow.models.evaluate()` standardizes evaluation across projects** so that a model logged today can be re-evaluated next year by someone who has never seen your `compute_metrics` helper.

> **MLflow version drift.** Older tutorials and blog posts call `mlflow.evaluate(...)`. As of MLflow 3.0 that top-level alias is **deprecated** in favor of `mlflow.models.evaluate(...)` — same signature, same return type, just a different import path. Use the `mlflow.models` form in new code.


## Prerequisites

**Diverges from upstream tutorial:**
Upstream's evaluation examples sometimes use `shap.datasets.adult()` (classification on the UCI Adult Set). We stay on the **California housing regression** problem from `c_`, `d_`, `e_` so the comparison with the hand-rolled metrics + plots in `d_` is apples-to-apples.

### Optional dependency: `shap`

`mlflow.models.evaluate(..., evaluators=["default"])` will automatically produce a **SHAP feature-importance** artifact *if `shap` is installed* (matplotlib is already a transitive dependency). If `shap` is missing, MLflow logs a warning, skips that artifact, and everything else still works. To get the full set of auto-generated plots, add it once:

```bash
uv add shap
```

(`pip` is forbidden in this repo — see `CLAUDE.md`. Commit `pyproject.toml` and `uv.lock` together.)

### Start the tracking server

**Missing from upstream tutorial:**
Before running any cell that calls `mlflow.set_tracking_uri(...)`, start the MLflow server **in a separate terminal** from the repo root:

```bash
mlflow ui --host 127.0.0.1 --port 5001
```

(Other notebooks in this repo also use port `5001` — see `c_logging_callbacks.ipynb`.)


In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.exceptions import RestException
from mlflow.metrics import MetricValue
from mlflow.models import MetricThreshold, infer_signature, make_metric

HOST = "127.0.0.1"
PORT = 5001
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

# Idempotent experiment creation — see principle 7 of the
# mlflow-tutorial-improve skill. Re-running this cell is a no-op.
experiment_name = "Model Evaluation"
try:
    mlflow.create_experiment(name=experiment_name)
except RestException as e:
    if "RESOURCE_ALREADY_EXISTS" not in str(e):
        raise
mlflow.set_experiment(experiment_name)


## Step 1: Load and prepare the data

California housing, same split as the earlier notebooks. The one new wrinkle: `mlflow.evaluate()` expects the held-out data as a **single DataFrame** that contains both the features and the target column — so we build `eval_data` by appending `y_test` onto a copy of `X_test`. The name of the target column is passed via the `targets=` argument later on.


In [ ]:
raw = fetch_california_housing(as_frame=True)
X, y = raw.data, raw.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

# Single frame: features + target column. The `targets="MedHouseVal"` argument
# to mlflow.evaluate tells it which column is the ground truth.
eval_data = X_test.copy()
eval_data["MedHouseVal"] = y_test

print(f"train: {X_train.shape},  eval: {eval_data.shape}")
eval_data.head(3)


## Step 2: Train a candidate model and log it

Same `RandomForestRegressor` pattern as `d_`, but with two details that matter for evaluation:

- **`infer_signature(...)`** records the input/output schema on the model artifact. Without it, `mlflow.evaluate()` still works, but the artifact is harder for downstream consumers to use — there's no contract for which columns the model expects.
- **`input_example=`** stores a few rows of representative input next to the model. The MLflow UI uses these to render a "try this model" panel.

We train **outside** the `with mlflow.start_run():` block (same pattern as `b_`): a training error never leaves a half-populated run on the server.


In [ ]:
regressor = RandomForestRegressor(n_estimators=120, max_depth=12, random_state=0)
regressor.fit(X_train, y_train)

signature = infer_signature(X_train, regressor.predict(X_train))


## Step 3: The minimal `mlflow.models.evaluate()` call — the beginner story

The cell below logs the trained model, then calls `mlflow.models.evaluate()` against `eval_data`. `evaluators=["default"]` selects MLflow's built-in evaluator, which for `model_type="regressor"` produces:

- **metrics**: `example_count`, `mean_absolute_error`, `mean_squared_error`, `root_mean_squared_error`, `r2_score`, `max_error`, `mean_absolute_percentage_error`, `sum_on_target`, `mean_on_target`.
- **artifacts**: `eval_results_table` (the per-row predictions + targets as a table), and — if `shap` is installed — `shap_feature_importance_plot`, `shap_beeswarm_plot`, `shap_summary_plot`.

Everything is logged onto the **currently active run**, so you put the `evaluate()` call inside a `start_run()` block.


In [ ]:
with mlflow.start_run(run_name="baseline_eval") as run:
    model_info = mlflow.sklearn.log_model(
        regressor,
        name="model",
        signature=signature,
        input_example=X_train.head(),
    )

    result = mlflow.models.evaluate(
        model=model_info.model_uri,
        data=eval_data,
        targets="MedHouseVal",
        model_type="regressor",
        evaluators=["default"],
    )

print("--- metrics ---")
for k, v in sorted(result.metrics.items()):
    print(f"  {k:35s} {v:.4f}")

print("\n--- artifacts ---")
for k, v in result.artifacts.items():
    print(f"  {k:35s} {v.uri}")


### What just happened

One call replaced what `d_`'s `objective` function did with five helper functions and a dozen `log_*` calls:

- The hand-rolled `compute_metrics(...)` → eight metrics, named consistently with everything else MLflow logs.
- The hand-rolled `plot_residuals` / `plot_predicted_vs_actual` / `plot_qq` → covered by `eval_results_table` (you can re-derive any plot from it) and, with `shap` installed, by global feature-importance plots that the hand-rolled approach didn't have at all.

The `result` object exposes everything programmatically too — `result.metrics`, `result.artifacts`, `result.tables["eval_results_table"]` — which is what makes this call usable from CI or from another notebook.


## Step 4: Custom metrics — `make_metric` and `extra_metrics` (advanced)

Built-in regression metrics treat every dollar of error the same and aggregate over the whole dataset. Stakeholders rarely think that way. A realtor cares about *"what fraction of homes did we price within ±$50K?"*; a portfolio manager cares about *"how often do we systematically under-quote?"*. RMSE answers neither directly.

`make_metric(eval_fn=..., greater_is_better=..., name=...)` plugs a Python function into `mlflow.models.evaluate()` as a first-class metric — it lands in `result.metrics` next to `root_mean_squared_error` and is searchable in the UI like any other metric.

The modern (MLflow 3.x) `eval_fn` signature is:

```python
def eval_fn(
    predictions: pd.Series,
    targets: pd.Series,
    metrics: dict[str, MetricValue],
) -> float | MetricValue:
    ...
```

`predictions` and `targets` are the columns the default evaluator already produced; `metrics` lets you derive new metrics from existing ones (e.g. RMSE-normalized-by-mean). Return either a plain float (single aggregate) or a `MetricValue` (one or more aggregates + optional per-row scores).

**Gotcha — don't add `**kwargs`.** The default evaluator inspects your `eval_fn`'s signature and tries to bind every parameter name to a column in the eval frame. A `**kwargs` parameter gets resolved as a required column named `kwargs`, which doesn't exist, and the run aborts with `Metric ... requires the following: missing columns ['kwargs']`. Stick to the three named parameters above.

**Naming convention for `MetricValue.aggregate_results`.** When you return a `MetricValue` with `aggregate_results={"within_50k_pct": ...}` for a metric named `within_50k`, MLflow flattens it into the run's metrics dict as the *combined* key `"within_50k/within_50k_pct"`. Use that combined key to look it up, and to reference it in `validation_thresholds` below.

The California housing target is in units of $100,000. *"Within $50K"* therefore means `|prediction − target| ≤ 0.5` — small enough to be operationally useful, large enough to be plausible on a noisy regressor.


In [ ]:
def within_50k(
    predictions: pd.Series,
    targets: pd.Series,
    metrics: dict[str, MetricValue],
) -> MetricValue:
    """Fraction of predictions within $50K (= 0.5 in target units) of the truth.

    Returns a MetricValue so MLflow can also surface per-row scores
    (1 = within tolerance, 0 = outside) on the eval_results_table.
    """
    absolute_error = (predictions - targets).abs()
    per_row = (absolute_error <= 0.5).astype(int)
    return MetricValue(
        scores=per_row.tolist(),
        aggregate_results={"within_50k_pct": float(per_row.mean())},
    )


within_50k_metric = make_metric(
    eval_fn=within_50k,
    greater_is_better=True,
    name="within_50k",
)


## Step 5: Custom artifacts — plots the default evaluator doesn't give you (advanced)

Default artifacts answer *"how does the model do overall?"*. Production monitoring usually needs the next question: *"where does it do badly?"* — by region, by feature bin, by class. A model with great average RMSE can still be systematically biased on the rich-or-poor tail.

`custom_artifacts=[fn, ...]` lets you attach arbitrary plots. The signature is:

```python
def custom_artifact(
    eval_df: pd.DataFrame,        # has "prediction" and "target" columns
    builtin_metrics: dict[str, float],
    artifacts_dir: str,           # temp dir; anything you save here gets logged
) -> dict[str, Any]:              # name -> path (or name -> figure)
    ...
```

The plot below bins the test set into `MedInc` deciles (median income of the block group) and plots **mean residual per decile**. A flat band at 0 = unbiased across income; a tilt = the model systematically over- or under-prices certain segments.

**Pattern: closure over the test features.** `eval_df` is guaranteed to contain only `prediction` and `target`; everything else (the `MedInc` column we want to bin on) lives in `X_test`. We close over `X_test` from the surrounding scope rather than re-deriving it. This is the cleanest way to give a custom artifact access to features the default evaluator hides.


In [ ]:
def residuals_by_income_decile(
    eval_df: pd.DataFrame,
    builtin_metrics: dict[str, float],
    artifacts_dir: str,
) -> dict[str, str]:
    """Mean residual per MedInc decile — exposes segment-level bias the default plots hide."""
    # eval_df rows correspond 1:1 with X_test rows; reset_index aligns them.
    medinc = X_test["MedInc"].reset_index(drop=True)
    residuals = (eval_df["target"] - eval_df["prediction"]).reset_index(drop=True)

    deciles = pd.qcut(medinc, q=10, labels=False, duplicates="drop")
    mean_residual = residuals.groupby(deciles).mean()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(mean_residual.index, mean_residual.values)
    ax.axhline(0, color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("MedInc decile (0 = poorest, 9 = richest)")
    ax.set_ylabel("mean residual (true − predicted)")
    ax.set_title("Segment-level bias: residuals by income decile")
    fig.tight_layout()

    path = os.path.join(artifacts_dir, "residuals_by_income_decile.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return {"residuals_by_income_decile": path}


## Step 6: Re-evaluate with the custom metric + custom artifact

We start a fresh run (so this evaluation is visible in the UI side-by-side with Step 3's baseline) and pass both `extra_metrics=[within_50k_metric]` and `custom_artifacts=[residuals_by_income_decile]`. The default evaluator still runs — `extra_metrics` and `custom_artifacts` *add to* the built-ins, they don't replace them.


In [ ]:
with mlflow.start_run(run_name="baseline_eval_with_extras") as run_with_extras:
    # Re-use the model logged in Step 3 by re-logging it here. (We could also
    # have skipped re-logging and passed the previous model_info.model_uri,
    # but re-logging keeps each run self-contained for the UI comparison view.)
    model_info = mlflow.sklearn.log_model(
        regressor,
        name="model",
        signature=signature,
        input_example=X_train.head(),
    )

    result_extras = mlflow.models.evaluate(
        model=model_info.model_uri,
        data=eval_data,
        targets="MedHouseVal",
        model_type="regressor",
        evaluators=["default"],
        extra_metrics=[within_50k_metric],
        custom_artifacts=[residuals_by_income_decile],
    )

# Note the combined "name/aggregate_key" form — see the naming-convention
# note in Step 4.
print(f"within_50k/within_50k_pct:  {result_extras.metrics['within_50k/within_50k_pct']:.3f}")
print(f"root_mean_squared_error:    {result_extras.metrics['root_mean_squared_error']:.4f}")
print("\nartifacts now include:", sorted(result_extras.artifacts))


## Step 7: Validation thresholds — evaluation as a promotion gate (advanced)

So far evaluation has been *descriptive* — numbers and plots a human reads. The MLOps superpower is making it *prescriptive*: **fail the run** if the model regresses past a threshold. Without this, the only thing standing between a bad model and production is whichever reviewer eyeballs the metrics in the UI.

`mlflow.validate_evaluation_results(...)` takes a dict of `MetricThreshold` objects and raises `ModelValidationFailedException` if the candidate doesn't meet them. Two modes:

- **Static threshold** — `MetricThreshold(threshold=0.6, greater_is_better=False)`: the candidate must clear an absolute bar.
- **Candidate-vs-baseline** — pass `baseline_result=...` and use `min_absolute_change` / `min_relative_change` to require the candidate beat the *current production model* by a margin. This is the realistic CI gate; we demo the static form for brevity and link to the candidate-vs-baseline pattern in Next steps.

Below we define a generous gate (RMSE ≤ 0.7, within-$50K ≥ 0.35) that the candidate should pass, then a tightened version that deliberately fails — wrap that call in `try/except` so the cell finishes and you can read the exception message.


In [ ]:
passing_thresholds = {
    "root_mean_squared_error": MetricThreshold(threshold=0.7, greater_is_better=False),
    "within_50k/within_50k_pct": MetricThreshold(threshold=0.35, greater_is_better=True),
}

mlflow.validate_evaluation_results(
    candidate_result=result_extras,
    baseline_result=None,  # static-threshold mode
    validation_thresholds=passing_thresholds,
)
print("Passing gate cleared.")

# Tighten one threshold past what this model actually achieved, to show
# what a failed gate looks like.
failing_thresholds = {
    "root_mean_squared_error": MetricThreshold(threshold=0.1, greater_is_better=False),
}

try:
    mlflow.validate_evaluation_results(
        candidate_result=result_extras,
        baseline_result=None,
        validation_thresholds=failing_thresholds,
    )
except Exception as exc:
    print(f"Gate failed as expected:\n  {type(exc).__name__}: {exc}")


### Why this matters

A passing gate is the precondition for an automated promotion in the registry. A typical CI flow looks like:

1. Train a candidate.
2. `mlflow.evaluate(...)` it against the current production eval set.
3. `mlflow.validate_evaluation_results(...)` against a baseline that is *the current production model*.
4. If it passes, `mlflow.register_model(...)` and (in MLflow 3.x) assign the `@production` alias.

This is the "what unlocks deployment" piece of the registry story — a future notebook will tie the registry alias workflow to this gate.


## Viewing the runs in the UI — what to click on

Open <http://127.0.0.1:5001/> and click into the **Model Evaluation** experiment. You should see two runs: `baseline_eval` (Step 3) and `baseline_eval_with_extras` (Step 6).

### Per-run views
- **Metrics tab** — both runs show the eight default regression metrics. `baseline_eval_with_extras` additionally shows `within_50k_pct`.
- **Artifacts tab** — both runs contain `eval_results_table.json` (every test row's prediction + target) and, if `shap` is installed, the SHAP plots. `baseline_eval_with_extras` additionally has `residuals_by_income_decile.png`.
- **Model artifact** — under `model/`, the logged sklearn model with its signature and `input_example.json`. This is what `mlflow.evaluate()` loaded back — evaluation is tied to the *logged* model, not the in-memory `regressor`.

### Side-by-side comparison
Select both runs in the experiment table and click **Compare**. The Parallel Coordinates and Scatter views show how the extras line up against the baseline metric set.


## Next steps

- **[`mlflow.evaluate()` API reference](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.evaluate)** — full parameter list, including `evaluator_config` (for SHAP tuning) and `predictions=` (for evaluating pre-computed predictions without re-running a model).
- **[Custom metrics & visualizations](https://mlflow.org/docs/latest/ml/evaluation/metrics-visualizations/)** — more `make_metric` patterns, including using `MetricValue.scores` to land per-row metric values in `eval_results_table`.
- **Candidate-vs-baseline validation** — the [`evaluate_with_model_validation.py`](https://github.com/mlflow/mlflow/blob/master/examples/evaluation/evaluate_with_model_validation.py) example in the MLflow repo shows the production pattern: log a *baseline* model (the current production model, or a `DummyRegressor` for the very first version), evaluate it, then validate the candidate against it with `min_absolute_change` / `min_relative_change`.
- **Tying evaluation to the registry** — once a candidate passes its gate, `mlflow.register_model(model_info.model_uri, "ca_housing_price")` plus a `@production` alias is what actually moves it into the serving path. A future notebook will cover the registry + alias workflow end-to-end.
- **GenAI evaluation is a separate world.** `mlflow.evaluate()` is for traditional ML. LLM and agent evaluation uses `mlflow.genai.evaluate()` with a different metric system (judges, traces). See the [GenAI evaluation docs](https://mlflow.org/docs/latest/genai/eval-monitor/) when you get there.
